# Bloomberg pull smoke test

Corre esto en el **mismo kernel** donde ya funciona `import bbgapi` (DataZone).

Orden: (1) `bbgapi` → (2) latest `bdp` → (3) history corta `bdh` → (4) escribir `sofr_live.csv` → (5) append live.

In [ ]:
import os, sys
from pathlib import Path

# <-- cambia si el repo está en otra ruta en DataZone
REPO = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/gh_fwd_screener")
if not REPO.exists():
    REPO = Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd:", Path.cwd())
print("python:", sys.executable)

In [ ]:
# 1) ¿Este kernel tiene bbgapi?
import bbgapi
from bbgapi import bquery
print("bbgapi OK", bbgapi)
print("bquery OK", bquery)

In [ ]:
# 2) Smoke crudo (1 ticker) — si esto falla, el resto también
import pandas as pd

spot = bquery.bdp(["USOSFR10 BGN Curncy"], ["PX_LAST"], request_reroute=True)
print("bdp:\n", spot)

hist1 = bquery.bdh(
    securities=["USOSFR10 BGN Curncy"],
    fields=["PX_LAST"],
    start_date="20260101",
    end_date=pd.Timestamp.today().strftime("%Y%m%d"),
    request_reroute=True,
)
print("bdh shape:", getattr(hist1, "shape", None))
print(hist1.tail())

In [ ]:
# 3) Universo del screener + latest via data.py
from data import _securities, _fetch_latest, _fetch_history, pull_history_into_store, append_live_bar, load_live_store
from curve_config import BBG_START_DATE, BBG_END_DATE, BBG_FREQ, LIVE_STORE_PATH

secs = _securities()
print(f"n_securities={len(secs)}")
print("first 5:", secs[:5])
print("last 5:", secs[-5:])
print(f"config history: {BBG_START_DATE} → {BBG_END_DATE or 'today'}  freq={BBG_FREQ}")
print(f"store: {LIVE_STORE_PATH}")

latest = _fetch_latest()
print("\nlatest shape:", latest.shape, "asof", latest.index[-1])
print(latest.T.head(10))

### 4) History corta (recomendado para la primera prueba)

La config pide `15min` desde `20240101` → puede ser enorme / lento. Empieza con **daily + ventana corta**.

In [ ]:
# History corta → escribe sofr_live.csv (sobrescribe el store)
hist = pull_history_into_store(
    start_date="20260101",   # acorta si quieres
    end_date="",             # hoy
    freq="1D",               # daily primero; luego prueba "15min"
)
print(hist.shape)
print(hist.index[0], "→", hist.index[-1])
print(hist.iloc[:, :5].tail())

In [ ]:
# 5) Append del print live (bdp) sobre el store
panel = append_live_bar()
print(panel.shape, "last=", panel.index[-1])
print(panel.iloc[:, :5].tail(3))

store = load_live_store()
print("\non disk:", LIVE_STORE_PATH, "n=", len(store))

### 6) (Opcional) History real de la config

Solo cuando el daily corto haya funcionado. Puede tardar mucho con `15min`.

In [ ]:
# Descomenta para tirar del window completo de curve_config
# hist_full = pull_history_into_store()  # usa BBG_START_DATE / BBG_FREQ
# print(hist_full.shape, hist_full.index[0], "→", hist_full.index[-1])

In [ ]:
# 7) Predicción encima del store (sin re-pull)
from run_prediction import run_once
table = run_once(refresh=False)
table[["model", "structure", "nivel", "y_hat_now", "residual", "y_hat_next"]].head(12)